In [1]:
import os, yaml, sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
from general_utils.utils import decode_matlab_strings

In [2]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    monkey_name: str = 'paul' # 'octavius'
    date: str =  '230204' #'220227to405' 
    img_size = 384
    folder_name = 'fewer_occlusion' # manyOO
    batch_size = 10
cfg = Cfg()

In [5]:

allimgs_path = f"{paths['livingstone_lab']}/tiziano/data/{cfg.monkey_name}_allimages{cfg.date}.mat"
with h5py.File(allimgs_path, "r") as f:
    try:
        refs = f["allimages"][:]      # shape (N, 1) of object refs
    except KeyError:
        refs = f["uniqueImage"][:]
    # end try:
    monkey_presentation_order = decode_matlab_strings(f, refs)
    monkey_presentation_order = sorted(set(monkey_presentation_order))


In [ ]:
import os
from torch.utils.data import Dataset
import torchvision
class FilteredImageFolder(Dataset):
    def __init__(self, imagefolder: torchvision.datasets.ImageFolder, subset_filenames: list[str]):
        self.imagefolder = imagefolder
        # Map: filename -> index in ImageFolder
        filename_to_index = {os.path.basename(path): i for i, (path, _) in enumerate(imagefolder.samples)} # saving all the filenames of my images in a dict {filename: label} ; imagefolder.samples is a list of tuples (fullpath, label) for all the images in the dataset
        # Preserve subset_filenames order
        self.indices = [filename_to_index[fname] for fname in subset_filenames if fname in filename_to_index] # stores the indices in the indicated order passed they will be used below to get_item
    # EOF
    def __len__(self):
        return len(self.indices)
    # EOF
    def __getitem__(self, idx):
        return self.imagefolder[self.indices[idx]] # it indexes the indices e.g. if self.indices = [40, 22, 3, ...], self.imagefolder[self.indices[1]] will yield self.imagefolder[22], i.e. the wanted (and ordered) image 
    # EOF
# EOC

In [35]:
from general_utils.utils import get_relevant_output_layers
from image_processing.utils import get_usual_transform
from torch.utils.data import DataLoader
import argparse
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = get_usual_transform(resize_size=cfg.img_size, normalize=False)
# task_list = get_relevant_output_layers(args.model_name, pkg=args.pkg)

base_dataset = ImageFolder(
    root=f"{paths['livingstone_lab']}/Stimuli/{cfg.folder_name}/",
    transform=transform,
    is_valid_file=lambda x: not x.endswith("Thumbs.db"), 
    allow_empty=True, 
)
dataset = FilteredImageFolder(
    base_dataset,
    subset_filenames=monkey_presentation_order[:100],
)
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False)


{'BigAnimate_01.jpg': 0, 'BigAnimate_02.jpg': 1, 'BigAnimate_03.jpg': 2, 'BigAnimate_04.jpg': 3, 'BigAnimate_05.jpg': 4, 'BigAnimate_06.jpg': 5, 'BigAnimate_07.jpg': 6, 'BigAnimate_08.jpg': 7, 'BigAnimate_09.jpg': 8, 'BigAnimate_10.jpg': 9, 'BigAnimate_11.jpg': 10, 'BigAnimate_12.jpg': 11, 'BigAnimate_13.jpg': 12, 'BigAnimate_14.jpg': 13, 'BigAnimate_15.jpg': 14, 'BigAnimate_16.jpg': 15, 'BigAnimate_17.jpg': 16, 'BigAnimate_18.jpg': 17, 'BigAnimate_19.jpg': 18, 'BigAnimate_20.jpg': 19, 'BigAnimate_21.jpg': 20, 'BigAnimate_22.jpg': 21, 'BigAnimate_23.jpg': 22, 'BigAnimate_24.jpg': 23, 'BigAnimate_25.jpg': 24, 'BigAnimate_26.jpg': 25, 'BigAnimate_27.jpg': 26, 'BigAnimate_28.jpg': 27, 'BigAnimate_29.jpg': 28, 'BigAnimate_30.jpg': 29, 'BigAnimate_31.jpg': 30, 'BigAnimate_32.jpg': 31, 'BigAnimate_33.jpg': 32, 'BigAnimate_34.jpg': 33, 'BigAnimate_35.jpg': 34, 'BigAnimate_36.jpg': 35, 'BigAnimate_37.jpg': 36, 'BigAnimate_38.jpg': 37, 'BigAnimate_39.jpg': 38, 'BigAnimate_40.jpg': 39, 'BigAnima

In [27]:
dataset = dataloader.dataset

for batch_idx, (images, labels) in enumerate(dataloader):
    start = batch_idx * dataloader.batch_size
    end = start + len(labels)

    batch_indices = dataset.indices[start:end]
    filenames = [
        os.path.basename(dataset.imagefolder.samples[i][0])
        for i in batch_indices
    ]

    print(f"batch {batch_idx}")
    print("files:", filenames)
    # ADD fix presentation order

batch 0
files: ['0-teak_tf53er.jpg', '07Model17RedSkiHelmet.jpg', '1054_detail.jpg', '1259_2341_popup1.jpg', '16234104.thl.jpg', '21SS8DBWDQL._SS500_.jpg', '26369935.thl.jpg', '26373229.thl.jpg', '26373355.thl.jpg', '26379142.thl.jpg']
batch 1
files: ['26381224.thl.jpg', '26387944.thl.jpg', '2sm.jpg', '31H95JKXWNL._SS500_.jpg', '35811374.thl.jpg', '54619901.jpg', '568eur6.jpg', '8027495.thl-1.jpg', '91180.jpg', 'ABEARMASK.jpg']
batch 2
files: ['AGARBACA8.jpg', 'AGARBCAN4.jpg', 'AJACKOL24.jpg', 'ALEI10.jpg', 'ALEI5.jpg', 'ALEI6.jpg', 'ALEI8.jpg', 'ALIONMASK.jpg', 'ALOCK6.jpg', 'AMICROS25.jpg']
batch 3
files: ['AMICROS34.jpg', 'AMICROS40.jpg', 'APACKME24.jpg', 'APALMTRE3.jpg', 'AREDROCK.jpg', 'ASALTPEPP.jpg', 'ASTOOL33.jpg', 'ASTOOL42.jpg', 'ASTRHATFL.jpg', 'ATABLE8.jpg']
batch 4
files: ['ATRUNK36.jpg', 'AYELLROCK.jpg', 'Aballofyarn3.jpg', 'Abird21.jpg', 'Aclock213.jpg', 'Alei22.jpg', 'Amicroscope23.jpg', 'Ascale120.jpg', 'Aseashe34.jpg', 'Astuffra9.jpg']
batch 5
files: ['Atent8.jpg', 'B